# ML-03 — Frame Your Lane as an ML Task

This notebook frames our content prioritization challenge as a formal machine learning task for **Lane 2: Refresh / Content Opportunity Scoring**.

> Work conducted using the standard FlyRank starter dataset.

## 1. My lane as an ML task (type)

### Task Type: Ranking / Scoring

- **Why:** The core decision is prioritizing which pages to review and refresh first. Since editor and content manager time is a scarce resource, we do not need a binary classification of every page. Instead, we need a priority score that sorts the pages from highest opportunity/risk to lowest.
- **Decision:** Which pages should a content manager or editor refresh today to prevent or reverse search traffic decline?
- **Action:** A content editor reviews the recommended page, updates content quality, optimizes titles/metadata, or merges thin/duplicate sections.
- **Wrong Call Cost:** Wasted hours spent reviewing pages that don't need updates (false positives), while high-value pages in active decline are missed (false negatives).

In [ ]:
# We map our goal to a Ranking/Scoring task, outputting a priority score for each page.
print("Task Frame: For content managers, deciding which pages to review first, we will build")
print("a scoring/ranking model from search and engagement signals, predicting/scoring decline risk")
print("measured by Precision@50. A wrong call costs wasted reviewer hours.")

## 2. Target or proxy

### Prediction Target: Content Decline

- **Target Label:** `is_declining_label = (trend_direction == "down")`
- **Where the label comes from:** It is an **observed outcome** in the starter dataset, computed from search impressions and clicks over a trailing 90-day window (i.e., whether the trend percentage is negative).
- **Honest Modeling Note:** To prevent future feature leakage in production, the model's features must be strictly computed from a historical feature window, while the target must be observed in a subsequent target window (e.g., prior 90 days features predicting the next 30 days decline).

In [ ]:
# The target label is derived from the observed direction of traffic trends
import os
import pandas as pd

paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    r"E:\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main\data\raw\content_refresh_anonymized.csv"
]
df = None
for path in paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        break

if df is not None:
    df['is_declining_label'] = df['trend_direction'] == 'down'
    base_rate = df['is_declining_label'].mean()
    print(f"Observed Base Rate of Decline: {base_rate:.1%}")
    print(f"Total Pages in Starter: {len(df):,}")
    print(f"Total Declining Pages: {df['is_declining_label'].sum():,}")

## 3. Success metric

### Metric: Precision@K (specifically Precision@50)

- **Defense:** Since content review capacity is limited (e.g., a reviewer handles 50 pages at a time), we care most about the precision of the top recommendations. Accuracy or AUC are less relevant than knowing: of the top 50 pages we recommended for refresh, how many actually needed it?
- **What is 'good':** The current hand-written rule baseline gets a Precision@50 of **0.240** (12 out of 50 correct). An ML model achieves a Precision@50 of **~0.740** (37 out of 50 correct) in the starter pipeline, representing a **3x lift** in efficiency. Any model scoring above **0.60** is considered highly successful.

In [ ]:
# Defining the target metrics
print("Baseline Rule Precision@50: 0.240")
print("Target ML Model Precision@50: >= 0.600 (aiming for 0.70+)")

## 4. The unit of analysis, as a real dataframe

### Unit of Analysis: A single pseudonymized content item (page)

- **Definition:** One row in the dataframe corresponds to one content item (`content_id`), filtered for valid exposure (`impressions_90d > 0` and `content_age_days >= 90`) and deduplicated.
- **Client Grounding:** Pages are associated with specific clients (`client_id`), which we will use to perform grouped train/test splits (ensuring no client's pages are in both train and test).

In [ ]:
if df is not None:
    # Filter rows based on standard starter rules
    filtered_df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"])
    
    print(f"Filtered Dataframe Shape: {filtered_df.shape}")
    print("\nSample Unit of Analysis (Page-level features):")
    print(filtered_df[["content_id", "client_id", "impressions_90d", "ctr", "avg_position", "is_declining_label"]].head(5).to_string(index=False))

## 5. Why ML beats a fixed rule here

### Why ML is Needed:

1. **Non-Linear Interactions:** Fixed rules use arbitrary thresholds (e.g., `avg_position <= 10` and `ctr < 0.5` or `days_since_last_update >= 180`). However, the interaction between position, impressions, intent, and age is highly non-linear and coupled.
2. **Client Variability:** FlyRank tracks 30+ different clients. A CTR of 0.5% might be excellent for one client/industry but extremely poor for another. Hardcoded thresholds cannot scale across different clients.
3. **Tangled Signals:** Factors such as intent, word count, search volume, and scroll rate shift dynamically. A machine learning model (like a random forest) can weight these features relative to each other based on actual outcome data, rather than human intuition.

In [ ]:
# Demonstrating the variability of CTR across different content types
if df is not None:
    ctr_by_type = filtered_df.groupby("content_type")["ctr"].agg(["mean", "median", "std", "count"])
    print("CTR Summary by Content Type (showing distinct distributions that a fixed rule ignores):")
    print(ctr_by_type.round(4).to_string())

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.